In [3]:
!pip install weaviate-client
!pip install datasets

/Users/jb/miniconda3/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=5875) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [1]:
# declare name of the collection
COLLECTION_NAME = "QuestionAnswering"
QUERY_SENTENCE = "사이보그가 뭐야?"
NUM_OF_DATA = 200
WEIGHT_OF_VECTOR_IN_HYBRID = 0.5

In [2]:
import pandas as pd
from datasets import load_dataset

DATA_SET    = "beomi/KoAlpaca-v1.1a"
DATA_FILE   = "./data/KoAlpaca-train.csv"

load_data = load_dataset(DATA_SET, split="train")
load_data.to_csv(DATA_FILE)
csv_data = pd.read_csv(DATA_FILE)
csv_data.head(5)
data_to_insert = csv_data.head(min(NUM_OF_DATA, len(load_data)))

Creating CSV from Arrow format:   0%|          | 0/22 [00:00<?, ?ba/s]

In [3]:
import weaviate

client = weaviate.connect_to_local(host='localhost', port=8080)

In [17]:
from weaviate.collections.classes.config import Tokenization
import weaviate.classes.config as wc

# Drop the collection
client.collections.delete(name=COLLECTION_NAME)

client.collections.create(
    name=COLLECTION_NAME,
    description=COLLECTION_NAME,
    properties=[
        wc.Property(
            name="instruction",
            data_type=wc.DataType.TEXT,
            tokenization=Tokenization.KAGOME_KR
        ),
        wc.Property(
            name="output",
            data_type=wc.DataType.TEXT,
            tokenization=Tokenization.KAGOME_KR
        ),
        wc.Property(
            name="url",
            data_type=wc.DataType.TEXT,
            skip_vectorization=True),
    ],
    vectorizer_config=wc.Configure.Vectorizer.text2vec_transformers(),
)

In [11]:
from weaviate.collections.classes.filters import Filter

target_collection = client.collections.get(COLLECTION_NAME)

# Fetch the first 200 rows from csv_data
data_to_insert = csv_data.head(min(NUM_OF_DATA, len(csv_data)))

with target_collection.batch.dynamic() as batch:
    for _, row in data_to_insert.iterrows():
        # 중복 확인
        response = target_collection.query.fetch_objects(
            filters=Filter.by_property("instruction").equal(row["instruction"]),
            limit=1
        )
        # object가 있으면 건너뛰기
        if response.objects:
            continue

        batch.add_object(
            properties={
                "instruction": row["instruction"],
                "output": row["output"],
                "url": row["url"],
            },
        )


In [12]:
from weaviate.classes.query import MetadataQuery

target_collection = client.collections.get(COLLECTION_NAME)

response = target_collection.query.near_text(
    query=QUERY_SENTENCE,
    limit=3,
    return_metadata=MetadataQuery(distance=True)
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)

{'instruction': '실랑이와 실갱이의 차이점은 무엇인가요?', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=13&dirId=130101&docId=88030440', 'output': '실갱이란 단어는 표준어가 아니기 때문에, 실랑이와의 차이점을 설명드리겠습니다. 실랑이와 승강이를 혼동하는 경우가 있습니다. 실랑이는 자신의 주장이나 생각이 상대방과 다르다고 이야기하며 상대방을 괴롭힘으로써 일방적으로 승리하려는 경향이 있습니다. 반면, 승강이는 서로 자기 주장을 고집하여 옥신각신하는 경우에 쓰이는 단어입니다. 양측이 팽팽하게 대립하는 경우에 승강이를 벌인다고 표현합니다.'}
0.22223186492919922
{'instruction': '전화기를 발명한 사람은 누구인가요? 그리고 그 사람에 대해 자세히 설명 부탁드립니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=111002&docId=61904077', 'output': '전화기를 발명한 사람은 1876년 알렉산더 그레이엄 벨이 발명했습니다. 하지만, 최초의 전화 발명자라는 주장은 독일인 과학자 필립 라이스가 1863년에 멀리 떨어진 두 장소에서 대화를 할 수 있는 텔레폰을 발명한 것이 밝혀졌습니다. 이것은 1963년 영국 스탠더드 텔레폰스 앤 케이블스(STC)사의 시험 결과로 확인되었습니다. STC사는 이 시기에 라이스가 고안한 장치로 여러 차례 시험을 실시해 성공을 거두었습니다. 이러한 사실이 뒤늦게 발각된 이유는 STC 회장이 라이스의 시험 결과가 미국의 벨사 입찰에 영향을 끼칠 것을 우려해 비밀로 묻어둔 것으로 추측됩니다.'}
0.2530491352081299
{'output': '계절마다 몸 색깔이 변하는 토끼는 북극에 사는 눈토끼 종류 뿐입니다. 이 종류는 여름에는 회갈색을 띄다가 겨울이 되면 흰색으로 변하는데, 이는 천적인 눈에 잘 눈치 채지 않도록 하는 보호색입니다.

In [15]:
target_collection = client.collections.get(COLLECTION_NAME)

response = target_collection.query.bm25(
    query=QUERY_SENTENCE,
    return_metadata=MetadataQuery(score=True),
    limit=3
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.score)

In [16]:
target_collection = client.collections.get(COLLECTION_NAME)

response = target_collection.query.hybrid(
    query=QUERY_SENTENCE,
    alpha=WEIGHT_OF_VECTOR_IN_HYBRID,
    return_metadata=MetadataQuery(score=True, explain_score=True),
    limit=3,
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.score, o.metadata.explain_score)

{'instruction': '실랑이와 실갱이의 차이점은 무엇인가요?', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=13&dirId=130101&docId=88030440', 'output': '실갱이란 단어는 표준어가 아니기 때문에, 실랑이와의 차이점을 설명드리겠습니다. 실랑이와 승강이를 혼동하는 경우가 있습니다. 실랑이는 자신의 주장이나 생각이 상대방과 다르다고 이야기하며 상대방을 괴롭힘으로써 일방적으로 승리하려는 경향이 있습니다. 반면, 승강이는 서로 자기 주장을 고집하여 옥신각신하는 경우에 쓰이는 단어입니다. 양측이 팽팽하게 대립하는 경우에 승강이를 벌인다고 표현합니다.'}
0.5 
Hybrid (Result Set vector,hybridVector) Document ca328228-179c-4991-a0a7-3f860e14edfa: original score 0.77776814, normalized score: 0.5
{'instruction': '전화기를 발명한 사람은 누구인가요? 그리고 그 사람에 대해 자세히 설명 부탁드립니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=111002&docId=61904077', 'output': '전화기를 발명한 사람은 1876년 알렉산더 그레이엄 벨이 발명했습니다. 하지만, 최초의 전화 발명자라는 주장은 독일인 과학자 필립 라이스가 1863년에 멀리 떨어진 두 장소에서 대화를 할 수 있는 텔레폰을 발명한 것이 밝혀졌습니다. 이것은 1963년 영국 스탠더드 텔레폰스 앤 케이블스(STC)사의 시험 결과로 확인되었습니다. STC사는 이 시기에 라이스가 고안한 장치로 여러 차례 시험을 실시해 성공을 거두었습니다. 이러한 사실이 뒤늦게 발각된 이유는 STC 회장이 라이스의 시험 결과가 미국의 벨사 입찰에 영향을 끼칠 것을 우려해 비밀로 묻어둔 것으로 추측됩니다.'}
0.2642923593521118 